# 4.2 Custom Model with Context Artifacts

**Objective:** Use MLflow model context to package a small artifact inside a custom model.

This notebook is fully independent. It does not depend on helper folders, custom utility modules, or files outside this notebook folder.

In [1]:

import os
import sys
import json
import time
import math
import pickle
import tempfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn
import mlflow.pyfunc
from mlflow.tracking import MlflowClient
from mlflow.models.signature import infer_signature

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    mean_absolute_error, mean_squared_error, r2_score
)
from sklearn.datasets import (
    load_iris, load_diabetes, load_digits, load_linnerud,
    load_wine, load_breast_cancer, make_classification, make_regression
)
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, ElasticNet
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

warnings.filterwarnings("ignore")


def find_project_root(start_path=None):
    """Find the mlflow_for_ml_dev folder from wherever Jupyter is started."""
    start = Path(start_path or Path.cwd()).resolve()
    for parent in [start] + list(start.parents):
        if parent.name == "mlflow_for_ml_dev" and (parent / "notebooks").exists():
            return parent
        if (parent / "datasets").exists() and (parent / "mlruns").exists() and (parent / "notebooks").exists():
            return parent
    # Fallback: if user started Jupyter inside notebooks subfolder
    for parent in [start] + list(start.parents):
        if parent.name == "notebooks":
            return parent.parent
    return start

PROJECT_ROOT = find_project_root()
DATASETS_DIR = PROJECT_ROOT / "datasets"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "artifacts_generated"

for folder in [DATASETS_DIR, MLRUNS_DIR, MODELS_DIR, ARTIFACTS_DIR, MLRUNS_DIR / ".trash"]:
    folder.mkdir(parents=True, exist_ok=True)

# IMPORTANT: Every notebook logs to the same mlruns folder inside this project.
mlflow.set_tracking_uri(MLRUNS_DIR.as_uri())
client = MlflowClient()

print("Project root      :", PROJECT_ROOT)
print("Datasets folder   :", DATASETS_DIR)
print("MLflow runs folder:", MLRUNS_DIR)
print("Models folder     :", MODELS_DIR)
print("Tracking URI      :", mlflow.get_tracking_uri())


Project root      : /Users/giridharsripathi/Documents/training/cgi_mlflow_new
Datasets folder   : /Users/giridharsripathi/Documents/training/cgi_mlflow_new/datasets
MLflow runs folder: /Users/giridharsripathi/Documents/training/cgi_mlflow_new/mlruns
Models folder     : /Users/giridharsripathi/Documents/training/cgi_mlflow_new/models
Tracking URI      : file:///Users/giridharsripathi/Documents/training/cgi_mlflow_new/mlruns


## 2. Custom pyfunc using model context artifacts

In [2]:
# Create a small lookup file that the model will read from MLflow artifacts.
lookup_path = ARTIFACTS_DIR / "thresholds.json"
with open(lookup_path, "w") as f:
    json.dump({"threshold": 60}, f)
print("Created artifact:", lookup_path)

Created artifact: /Users/giridharsripathi/Documents/training/cgi_mlflow_new/notebooks/artifacts_generated/thresholds.json


In [3]:
import mlflow.pyfunc

class ThresholdModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        with open(context.artifacts["thresholds"], "r") as f:
            self.config = json.load(f)

    def predict(self, context, model_input):
        threshold = self.config["threshold"]
        return np.where(model_input["score"] >= threshold, "pass", "review")

mlflow.set_experiment("independent_04_custom_pyfunc_context")
with mlflow.start_run(run_name="custom_model_with_context") as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=ThresholdModel(),
        artifacts={"thresholds": str(lookup_path)}
    )
    model_uri = f"runs:/{run.info.run_id}/model"

loaded_model = mlflow.pyfunc.load_model(model_uri)
print(loaded_model.predict(pd.DataFrame({"score": [40, 65, 90]})))

/Users/giridharsripathi/Documents/training/cgi_mlflow_new/.venv/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:134: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/05/12 13:12:27 INFO mlflow.tracking.fluent: Experiment with name 'independent_04_custom_pyfunc_context' does not exist. Creating a new experiment.
2026/05/12 13:12:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


['review' 'pass' 'pass']
